## Carga del modelo

In [9]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("../data/processed_features_v2.csv")

## Target definition 

In [10]:
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date"])
y = df[targets]

## Baseline random forest

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

rf_baseline = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_baseline.fit(X_train, y_train)

y_pred_rf = rf_baseline.predict(X_test)

rf_baseline_score = r2_score(y_test, y_pred_rf)
print("RF baseline R²:", rf_baseline_score)

RF baseline R²: 0.808361319742013


## Feature importance

In [12]:
importances = rf_baseline.feature_importances_
feat_importance = pd.Series(importances, index=X.columns)
print(feat_importance.sort_values(ascending=False).head(10))

Longitude             0.454853
Latitude              0.276884
pet                   0.032248
pet_day               0.018655
dayofyear             0.017649
swir22                0.015912
ndmi_day              0.013018
green                 0.012631
swir22_green_ratio    0.012374
swir_ratio            0.012108
dtype: float64


## Random Forest sin geo

In [13]:
X_no_geo = X.drop(columns=["Latitude", "Longitude"])

X_train_ng, X_test_ng, y_train_ng, y_test_ng = train_test_split(
    X_no_geo, y,
    test_size=0.2,
    random_state=42
)

rf_no_geo = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_no_geo.fit(X_train_ng, y_train_ng)

y_pred_ng = rf_no_geo.predict(X_test_ng)

rf_no_geo_score = r2_score(y_test_ng, y_pred_ng)
print("RF no geo R²:", rf_no_geo_score)

RF no geo R²: 0.5971981863690713


## Gradient Boosting

In [14]:
gbr = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
)

gbr.fit(X_train_ng, y_train_ng)
y_pred_gbr = gbr.predict(X_test_ng)

gbr_score = r2_score(y_test_ng, y_pred_gbr)
print("GradientBoosting R²:", gbr_score)

GradientBoosting R²: 0.371016761139585


## Random Forest Ajustado

In [15]:
rf_tuned = RandomForestRegressor(
    n_estimators=400,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train_ng, y_train_ng)
y_pred_rf_tuned = rf_tuned.predict(X_test_ng)

rf_tuned_score = r2_score(y_test_ng, y_pred_rf_tuned)
print("RF tuned R²:", rf_tuned_score)

RF tuned R²: 0.5187911193369289


## Comparacion final de modelos

In [16]:
print("RF baseline:", rf_baseline_score)
print("RF no geo:", rf_no_geo_score)
print("GradientBoosting:", gbr_score)
print("RF tuned:", rf_tuned_score)

RF baseline: 0.808361319742013
RF no geo: 0.5971981863690713
GradientBoosting: 0.371016761139585
RF tuned: 0.5187911193369289
